In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

fatal: destination path 'CTAB-GAN-Plus' already exists and is not an empty directory.


In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
secondary_mushroom = fetch_ucirepo(id=848)

# data (as pandas dataframes)
X = secondary_mushroom.data.features
y = secondary_mushroom.data.targets

# metadata
print(secondary_mushroom.metadata)

# variable information
print(secondary_mushroom.variables)

mushroom_data = pd.concat([X, y], axis=1)

target_col = "class"

CONTINUOUS_COLS = ["cap-diameter", "stem-height", "stem-width"]
CATEGORICAL_COLS = [
    col for col in mushroom_data.columns if col not in CONTINUOUS_COLS
]

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

# Fast dev mode: fewer epochs for generators only (set False for full paper run)
FAST_MODE = True
CTABGAN_EPOCHS = 10 if FAST_MODE else 150
WGAN_EPOCHS = 10 if FAST_MODE else 100
SDV_EPOCHS = 10 if FAST_MODE else 300
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
# All 6 synthetic generators for TSTR evaluation
GENERATORS_TO_EVAL = [
    "CTGAN", "CopulaGAN", "TVAE", "GaussianCopula", "WGAN_GP", "CTABGAN"
]

# Stratified subsample (full dataset has 61069 rows)
_, mushroom_data = train_test_split(
    mushroom_data,
    train_size=N_SAMPLES,
    stratify=mushroom_data[target_col],
    random_state=SEED,
)
mushroom_data = mushroom_data.reset_index(drop=True)

# Handle missing values
numeric_cols = mushroom_data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = [col for col in CATEGORICAL_COLS if col in mushroom_data.columns]

for col in numeric_cols:
    mushroom_data[col] = pd.to_numeric(mushroom_data[col], errors="coerce")
    mushroom_data[col] = mushroom_data[col].fillna(mushroom_data[col].median())

for col in categorical_cols:
    fill = mushroom_data[col].mode().iloc[0] if not mushroom_data[col].mode().empty else "missing"
    mushroom_data[col] = mushroom_data[col].fillna(fill)

# Label-encode categorical columns (ordinal encoding)
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    mushroom_data[col] = le.fit_transform(mushroom_data[col].astype(str))
    label_encoders[col] = le

# Encoded positive class for binary metrics (poisonous = "p")
pos_label = int(label_encoders[target_col].transform(["p"])[0])

print(f"Encoded {len(categorical_cols)} categorical columns.")
print(f"{target_col} mapping: {dict(zip(label_encoders[target_col].classes_, range(len(label_encoders[target_col].classes_))))}")
print(f"pos_label (encoded poisonous class): {pos_label}")

X = mushroom_data.drop(columns=[target_col])
y = mushroom_data[target_col]

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(mushroom_data)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


{'uci_id': 848, 'name': 'Secondary Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/848/secondary+mushroom+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/848/data.csv', 'abstract': 'Dataset of simulated mushrooms for binary classification into edible and poisonous.', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 61068, 'num_features': 20, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2021, 'last_updated': 'Wed Apr 10 2024', 'dataset_doi': '10.24432/C5FP5Q', 'creators': ['Dennis Wagner', 'D. Heider', 'Georges Hattab'], 'intro_paper': {'ID': 259, 'type': 'NATIVE', 'title': 'Mushroom data creation, curation, and simulation to support classification tasks', 'authors': 'Dennis Wagner, D. Heider, Georges Hattab', 'venue': 'Scientific Reports', 'year': 2021, 'journal': None, '

In [3]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

CONTINUOUS_COLS = ["cap-diameter", "stem-height", "stem-width"]
CATEGORICAL_COLS = [
    col for col in mushroom_data.columns if col not in CONTINUOUS_COLS
]

# Re-apply encoding if the load cell was not re-run after edits
if mushroom_data[target_col].dtype == "object":
    numeric_cols = mushroom_data.select_dtypes(include=["int64", "float64"]).columns
    for col in numeric_cols:
        mushroom_data[col] = pd.to_numeric(mushroom_data[col], errors="coerce")
        mushroom_data[col] = mushroom_data[col].fillna(mushroom_data[col].median())
    for col in CATEGORICAL_COLS:
        fill = mushroom_data[col].mode().iloc[0] if not mushroom_data[col].mode().empty else "missing"
        mushroom_data[col] = mushroom_data[col].fillna(fill)
    label_encoders = {}
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        mushroom_data[col] = le.fit_transform(mushroom_data[col].astype(str))
        label_encoders[col] = le
    pos_label = int(label_encoders[target_col].transform(["p"])[0])
elif "label_encoders" not in globals():
    label_encoders = {}
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        le.fit(mushroom_data[col].astype(str))
        label_encoders[col] = le
    try:
        pos_label = int(label_encoders[target_col].transform(["p"])[0])
    except ValueError:
        pos_label = int(mushroom_data[target_col].max())

# ---------------------------------------------------
# GENERATOR TRAINING DATA (no stratified split)
# ---------------------------------------------------

train_real = mushroom_data.copy()
test_real = mushroom_data.copy()

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# CTABGAN
# ---------------------------------------------------

try:

    data_path = "secondary_mushroom_train.csv"
    train_real.to_csv(data_path, index=False)

    categorical_columns = CATEGORICAL_COLS

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=categorical_columns,
        log_columns=[],
        mixed_columns={},
        integer_columns=CONTINUOUS_COLS,
        problem_type={"Classification": target_col}
    )

    ctabgan.synthesizer.epochs = CTABGAN_EPOCHS
    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    for col in CATEGORICAL_COLS:
        if col in synthetic_ctabgan.columns:
            col_max = int(train_real[col].max())
            synthetic_ctabgan[col] = pd.to_numeric(
                synthetic_ctabgan[col], errors="coerce"
            )
            synthetic_ctabgan[col] = synthetic_ctabgan[col].fillna(
                train_real[col].mode()[0]
            )
            synthetic_ctabgan[col] = (
                synthetic_ctabgan[col]
                .round()
                .clip(0, col_max)
                .astype(int)
            )

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)



================ SINGLE RUN ================


100%|██████████| 10/10 [2:15:03<00:00, 810.36s/it]


Finished training in 8120.825145244598  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 50.36it/s]|
Column Shapes Score: 92.86%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:04<00:00, 42.58it/s]|
Column Pair Trends Score: 83.15%

Overall Score (Average): 88.0%

CTABGAN: 0.88


In [4]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(WGAN_EPOCHS):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    for col in CATEGORICAL_COLS:
        col_max = int(train_real[col].max())
        synthetic_wgan[col] = (
            synthetic_wgan[col]
            .round()
            .clip(0, col_max)
            .astype(int)
        )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 57.07it/s]|
Column Shapes Score: 89.98%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:04<00:00, 43.75it/s]|
Column Pair Trends Score: 72.2%

Overall Score (Average): 81.09%

WGAN_GP: 0.8109


In [5]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "TVAE": TVAESynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata),
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 54.58it/s]|
Column Shapes Score: 92.78%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:04<00:00, 45.88it/s]|
Column Pair Trends Score: 84.12%

Overall Score (Average): 88.45%

CTGAN: 0.8845
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 137.45it/s]|
Column Shapes Score: 89.72%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:01<00:00, 127.51it/s]|
Column Pair Trends Score: 79.65%

Overall Score (Average): 84.68%

CopulaGAN: 0.8468
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 165.22it/s]|
Column Shapes Score: 87.33%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:01<00:00, 136.24it/s]|
Column Pair Trends Score: 71.76%

Overall Score (Average): 79.55%

TVAE: 0.7955
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 179.14it/s]|
Column Sha

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(max_iter=500, solver="liblinear", random_state=42),
        "SVM-RBF": LinearSVC(max_iter=500, dual="auto", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=12),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(max_iter=5000, solver="liblinear", random_state=42),
        "SVM-RBF": SVC(kernel="rbf", cache_size=1000, tol=1e-3, random_state=42),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")


Classifier evaluation: 10 models, 10 seeds
FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd


In [8]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


Skipping duplicate TRTR cell. Run the comparison cell for TRTR/TSTR (10 models, 10 seeds).


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=_safe_stratify(y_train),
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=_safe_stratify(y_test),
            )

            scaler = StandardScaler().fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)
            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train_s, y_train)
            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)


In [10]:
import pandas as pd

label_col = "class"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real)")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Generators: {len(model_order)}"
)
print(f"Classifier models: {list(models.keys())}")
print(f"Synthetic generators: {model_order}")

trtr_results = evaluate_models(
    train_df=mushroom_data,
    test_df=mushroom_data,
    label="class",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=mushroom_data,
        label="class",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)
Classifiers: 10 | Seeds: 10 | Generators: 6
Classifier models: ['LogReg', 'SVM-RBF', 'KNN', 'NaiveBayes', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoost', 'AdaBoost', 'MLP']
Synthetic generators: ['CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'WGAN_GP', 'CTABGAN']
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0001
5,RandomForest,0.9999 ± 0.0000,0.9999 ± 0.0000,0.9999 ± 0.0001,0.9999 ± 0.0001
9,MLP,0.9998 ± 0.0001,0.9998 ± 0.0001,0.9998 ± 0.0001,0.9998 ± 0.0002
2,KNN,0.9994 ± 0.0002,0.9994 ± 0.0002,0.9994 ± 0.0002,0.9994 ± 0.0002
4,DecisionTree,0.9431 ± 0.0031,0.9486 ± 0.0027,0.9504 ± 0.0048,0.9469 ± 0.0043
7,GradientBoost,0.8177 ± 0.0095,0.8351 ± 0.0099,0.8381 ± 0.0070,0.8325 ± 0.0191
0,LogReg,0.6533 ± 0.0034,0.7102 ± 0.0025,0.6623 ± 0.0032,0.7656 ± 0.0028
8,AdaBoost,0.6530 ± 0.0091,0.6587 ± 0.0100,0.7262 ± 0.0216,0.6037 ± 0.0233
1,SVM-RBF,0.6527 ± 0.0035,0.7112 ± 0.0025,0.6603 ± 0.0033,0.7705 ± 0.0027
3,NaiveBayes,0.6094 ± 0.0022,0.5472 ± 0.0039,0.7669 ± 0.0027,0.4253 ± 0.0046


CTGAN - TSTR
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
7,GradientBoost,0.5636 ± 0.0165,0.6165 ± 0.0214,0.6017 ± 0.0157,0.6338 ± 0.0430
8,AdaBoost,0.5610 ± 0.0233,0.5853 ± 0.0277,0.6163 ± 0.0277,0.5604 ± 0.0531
0,LogReg,0.5579 ± 0.0126,0.6029 ± 0.0089,0.6017 ± 0.0152,0.6050 ± 0.0211
1,SVM-RBF,0.5578 ± 0.0129,0.6032 ± 0.0092,0.6014 ± 0.0154,0.6059 ± 0.0214
3,NaiveBayes,0.5532 ± 0.0110,0.5622 ± 0.0232,0.6185 ± 0.0249,0.5196 ± 0.0517
6,ExtraTrees,0.5476 ± 0.0179,0.5761 ± 0.0239,0.5996 ± 0.0152,0.5549 ± 0.0338
5,RandomForest,0.5433 ± 0.0176,0.5748 ± 0.0164,0.5951 ± 0.0193,0.5566 ± 0.0246
4,DecisionTree,0.5328 ± 0.0275,0.5682 ± 0.0426,0.5820 ± 0.0222,0.5584 ± 0.0693
9,MLP,0.5180 ± 0.0197,0.5202 ± 0.0201,0.5815 ± 0.0235,0.4711 ± 0.0240
2,KNN,0.4929 ± 0.0125,0.4851 ± 0.0170,0.5559 ± 0.0152,0.4309 ± 0.0237


CTGAN - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
3,NaiveBayes,0.5852 ± 0.0084,0.7091 ± 0.0103,0.5807 ± 0.0073,0.9120 ± 0.0398
0,LogReg,0.5837 ± 0.0083,0.7157 ± 0.0066,0.5763 ± 0.0047,0.9443 ± 0.0161
1,SVM-RBF,0.5837 ± 0.0079,0.7162 ± 0.0062,0.5760 ± 0.0045,0.9467 ± 0.0154
8,AdaBoost,0.5573 ± 0.0127,0.7020 ± 0.0133,0.5603 ± 0.0077,0.9410 ± 0.0453
7,GradientBoost,0.5484 ± 0.0149,0.6924 ± 0.0136,0.5564 ± 0.0077,0.9169 ± 0.0318
5,RandomForest,0.5413 ± 0.0155,0.6477 ± 0.0132,0.5647 ± 0.0118,0.7604 ± 0.0318
6,ExtraTrees,0.5380 ± 0.0104,0.6415 ± 0.0112,0.5632 ± 0.0062,0.7453 ± 0.0208
2,KNN,0.5353 ± 0.0107,0.6041 ± 0.0141,0.5729 ± 0.0085,0.6394 ± 0.0277
9,MLP,0.5295 ± 0.0108,0.6002 ± 0.0117,0.5679 ± 0.0087,0.6366 ± 0.0197
4,DecisionTree,0.5100 ± 0.0362,0.5685 ± 0.0637,0.5529 ± 0.0288,0.5929 ± 0.1064


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,CTABGAN,ExtraTrees,0.462003,0.358470,0.436802,0.254672,1.0000 ± 0.0000,0.5380 ± 0.0104
1,CTABGAN,RandomForest,0.458548,0.352187,0.435239,0.239478,0.9999 ± 0.0000,0.5413 ± 0.0155
2,CTABGAN,MLP,0.470318,0.399668,0.431967,0.363207,0.9998 ± 0.0001,0.5295 ± 0.0108
3,CTABGAN,KNN,0.464034,0.395364,0.426498,0.360027,0.9994 ± 0.0002,0.5353 ± 0.0107
4,CTABGAN,DecisionTree,0.433120,0.380138,0.397431,0.354042,0.9431 ± 0.0031,0.5100 ± 0.0362
5,CTABGAN,GradientBoost,0.269361,0.142709,0.281631,-0.084386,0.8177 ± 0.0095,0.5484 ± 0.0149
6,CTABGAN,LogReg,0.069527,-0.005497,0.086023,-0.178716,0.6533 ± 0.0034,0.5837 ± 0.0083
7,CTABGAN,AdaBoost,0.095730,-0.043328,0.165871,-0.337348,0.6530 ± 0.0091,0.5573 ± 0.0127
8,CTABGAN,SVM-RBF,0.068986,-0.005036,0.084322,-0.176241,0.6527 ± 0.0035,0.5837 ± 0.0079
9,CTABGAN,NaiveBayes,0.024139,-0.161893,0.186283,-0.486651,0.6094 ± 0.0022,0.5852 ± 0.0084


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
5,WGAN_GP,0.168952,0.130843,0.181303,0.084987
4,TVAE,0.229401,0.199914,0.216600,0.192917
3,GaussianCopula,0.259187,0.206670,0.255173,0.164687
2,CopulaGAN,0.280621,0.168226,0.294809,-0.002443
0,CTABGAN,0.281576,0.181278,0.293207,0.030808
1,CTGAN,0.290006,0.271565,0.264974,0.284704


In [11]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")


Results saved to: TRTR_TSTR_results.xlsx
